In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# weight_height (신체계측정보) 환자 수 계산
import pandas as pd
import glob
import os

processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Weight-and-Height/de')
processed_files = glob.glob(os.path.join(processed_table_dir, '*.csv')) # 가명화 처리 완료된 신체계측정보 파일 목록

patient_ids = set()

for file in processed_files:
    try:
        table_df = pd.read_csv(file, dtype = {'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        if 'pt_id' in table_df.columns:
            unique_ids = table_df['pt_id'].dropna().unique() # 파일별 환자 고윳값 저장
            patient_ids.update(unique_ids)
    except Exception as e:
        print(f"파일 처리 오류: {file} -> {e}")

print(f"de 폴더 내 pt_id 고유값 수: {len(patient_ids):,}명")


In [ ]:
import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Weight-and-Height/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "*.csv"))

# 결과 저장
results = []

# 고유 환자 ID 저장
unique_pt_ids = set()


for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype={'weight': float, 'height': float})

        # ① 환자 수 (고유 pt_id)
        if 'pt_id' in table_df.columns:
            patient_ids = table_df['pt_id'].dropna().unique()
            unique_pt_ids.update(patient_ids)


        # 결측값 제거
        weight_vals = table_df['weight'].dropna()
        height_vals = table_df['height'].dropna()

        # 5의 배수 여부 확인
        weight_check = all((weight_vals % 5 == 0))
        height_check = all((height_vals % 5 == 0))

        # 결과 저장
        result = {
            'file': os.path.basename(file),
            'total_weight_values': len(weight_vals),
            'total_height_values': len(height_vals),
            'weight_all_multiple_of_5': weight_check,
            'height_all_multiple_of_5': height_check
        }
        results.append(result)

        print(f"✅ {file}: weight 5의 배수 = {weight_check}, height 5의 배수 = {height_check}")

    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")

# 데이터프레임으로 결과 정리
rare_combinations_df = pd.DataFrame(results)
print(f" - 고유 환자 수: {len(unique_pt_ids)}")
print("\n📊 전체 결과 요약:")
print(rare_combinations_df)


In [ ]:
# Basic info 확인

import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Basic-Info/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "*.csv"))

results = []

for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype=str)

        # 환자 수 계산
        n_unique_pt = table_df['pt_id'].nunique() if 'pt_id' in table_df.columns else None

        # birth_date 컬럼 day == 1 여부
        birth_check = False
        if 'birth_date' in table_df.columns:
            birth_dates = pd.to_datetime(table_df['birth_date'], errors='coerce')
            valid_dates = birth_dates.dropna()
            birth_check = all(valid_dates.dt.day == 1)
        else:
            print(f"⚠️ {file}: birth_date 컬럼 없음")

        # death_cause_cd / death_cause_nm 최소 빈도
        death_min_combo = None
        if {'death_cause_cd', 'death_cause_nm'}.issubset(table_df.columns):
            combination_counts_df = table_df.groupby(['death_cause_cd', 'death_cause_nm']).size().reset_index(name = 'count')
            if not combination_counts_df.empty:
                min_count = combination_counts_df['count'].min()
                death_min_combo = combination_counts_df[combination_counts_df['count'] == min_count].to_dict(orient = 'records')
            else:
                death_min_combo = []

        # nation_cd / nation_nm 고유값 확인
        unique_nation_cd = table_df['Region_cd'].dropna().unique().tolist() if 'Region_cd' in table_df.columns else []
        unique_nation_nm = table_df['Region_nm'].dropna().unique().tolist() if 'Region_nm' in table_df.columns else []

        results.append({
            'file': os.path.basename(file),
            'n_unique_pt': n_unique_pt,
            'birth_date_all_day_1': birth_check,
            'death_min_combo': death_min_combo,
            'nation_cd_unique': unique_nation_cd,
            'nation_nm_unique': unique_nation_nm
        })

        print(f"\n📄 {os.path.basename(file)}")
        print(f" - 고유 환자 수: {n_unique_pt}")
        print(f" - birth_date day==1? → {birth_check}")
        print(f" - death_cause 최소 등장 빈도:")
        for combo in death_min_combo:
            print(f"  -> {combo['death_cause_cd']} / {combo['death_cause_nm']}: {combo['count']}건")
        print(f" - 고유 nation_cd → {unique_nation_cd}")
        print(f" - 고유 nation_nm → {unique_nation_nm}")

    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")

# 결과 요약
rare_combinations_df = pd.DataFrame(results)
print("\n📊 전체 결과 요약:")
print(rare_combinations_df)


In [ ]:
# order_civ1 확인

import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Order-CIV1/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "*.csv"))

# 고유 환자 ID 저장
unique_pt_ids = set()

# 조건 플래그 (투약체중 소수점 0인지, 항정신성약물(N05A)값이 없는지)
all_med_weight_decimal_only_0 = True
no_N05A_found = True

for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        # ① 환자 수 (고유 pt_id)
        if 'pt_id' in table_df.columns:
            patient_ids = table_df['pt_id'].dropna().unique()
            unique_pt_ids.update(patient_ids)

        # ② med_weight 소수점이 0이 아닌 값 확인
        if all_med_weight_decimal_only_0 and 'med_weight_value' in table_df.columns:
            med_weight_series = table_df['med_weight_value'].dropna()
            med_weight_float = pd.to_numeric(med_weight_series, errors = 'coerce')
            if ((med_weight_float % 1) != 0).any():
                all_med_weight_decimal_only_0 = False

        # ③ N05A로 시작하는 값 확인
        def starts_with_N05A(series):
            return series.dropna().str.startswith("N05A").any()

        if no_N05A_found:
            for col in 'order_cd':
                if col in table_df.columns and has_N05A(table_df[col]):
                    no_N05A_found = False
                    break

    except Exception as e:
        print(f"오류 발생 - {file}: {e}")

# 출력
print("\n [종합 결과]")
print(f" - 고유 환자 수: {len(unique_pt_ids)}")
print(f" - med_weight 값 소수점 아래가 전부 0 → {all_med_weight_decimal_only_0}")
print(f" - 처방코드에 N05A로 시작하는 값 없음 → {no_N05A_found}")


In [ ]:
# nursing_initial 학인

import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Nursing-Initial-Assessment/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "*.csv"))

# 환자 번호 고유값 저장용
unique_pt_ids = set()

# 플래그와 디버깅용 세부정보 저장
obgyn_found = False
psych_found = False

obgyn_details = []
psych_details = []

min_diagnosis_freq = None

for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype=str)
        filename = os.path.basename(file)

        if 'pt_id' in table_df.columns:
            patient_ids = table_df['pt_id'].dropna().unique()
            unique_pt_ids.update(patient_ids)

        for col in ['opd_nm', 'depart', 'ad_ward']:
            if col in table_df.columns:
                col_series = table_df[col].dropna()

                # 산부인과
                obgyn_match = col_series[col_series.str.contains("산부인과", regex=False)]
                if not obgyn_match.empty:
                    obgyn_found = True
                    obgyn_details.append({
                        'file': filename,
                        'column': col,
                        'examples': obgyn_match.unique()[:3].tolist()
                    })

                # 정신과
                psych_match = col_series[col_series.str.contains("정신건강의학과|정신의학과", regex=True)]
                if not psych_match.empty:
                    psych_found = True
                    psych_details.append({
                        'file': filename,
                        'column': col,
                        'examples': psych_match.unique()[:3].tolist()
                    })

        # 상병명 최소 등장 빈도
        if 'dig_nm' in table_df.columns:
            dig_counts = table_df['dig_nm'].dropna().value_counts()
            if not dig_counts.empty:
                current_min = dig_counts.min()
                if min_diagnosis_freq is None or current_min < min_diagnosis_freq:
                    min_diagnosis_freq = current_min

    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")

# ✅ 최종 결과 출력
print("\n✅ [nursing_initial 폴더 종합 결과]")
print(f'환자 수: {len(unique_pt_ids)}')
print(f"산부인과 관련 내용 존재 여부 → {obgyn_found}")
print(f"정신과 관련 내용 존재 여부 → {psych_found}")
print(f"상병명 최소 등장 빈도수 → {min_diagnosis_freq if min_diagnosis_freq is not None else '데이터 없음'}")

# 세부 확인
if obgyn_found:
    print("\n📌 산부인과 관련 발견된 위치:")
    for entry in obgyn_details:
        print(f" - 파일: {entry['file']} / 컬럼: {entry['column']} / 예시: {entry['examples']}")

if psych_found:
    print("\n📌 정신과 관련 발견된 위치:")
    for entry in psych_details:
        print(f" - 파일: {entry['file']} / 컬럼: {entry['column']} / 예시: {entry['examples']}")


In [ ]:
# order_civ2 확인

import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Order-CIV2/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "*.csv"))

# pt_id 전체 고유값 저장용 집합
unique_pt_ids = set()

# N05A 관련 여부 플래그
no_N05A_found = True

for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        # ① pt_id 수집
        if 'pt_id' in table_df.columns:
            patient_ids = table_df['pt_id'].dropna().unique()
            unique_pt_ids.update(patient_ids)

        # ② med_cd에 N05A 시작 여부 확인
        if no_N05A_found and 'med_cd' in table_df.columns:
            if table_df['med_cd'].dropna().str.startswith("N05A").any():
                no_N05A_found = False  # 하나라도 있으면 False

    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")

# ✅ 최종 결과 출력
print("\n✅ [order_civ2 종합 결과]")
print(f"전체 고유 환자 수: {len(unique_pt_ids)}")
print(f"med_cd 컬럼에 N05A로 시작하는 값이 없는가? → {no_N05A_found}")


In [ ]:
# Dx_SNOMED 확인

import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Dx-SNOMED/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "**/*.csv"), recursive = True)

# 전체 고유 환자 수 저장용 집합
unique_pt_ids = set()

# 산부인과/정신과 관련 키워드 존재 여부
no_obgyn_or_psych = True  # True면 없는 상태

# 검사할 컬럼과 키워드
target_cols = ['depart', 'ad_depart_nm', 'dc_depart_nm']
keywords = "산부인과|정신건강의학과"

# 파일 순회
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        # pt_id 수집
        if 'pt_id' in table_df.columns:
            unique_pt_ids.update(table_df['pt_id'].dropna().unique())

        # 키워드 포함 여부 확인
        if no_obgyn_or_psych:
            for col in target_cols:
                if col in table_df.columns:
                    if table_df[col].fillna('').str.contains(keywords, regex=True).any():
                        no_obgyn_or_psych = False
                        break  # 하나라도 있으면 중단

    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")

# ✅ 최종 결과 출력
print("\n✅ [Dx_SNOMED 종합 결과]")
print(f"전체 고유 환자 수: {len(unique_pt_ids)}")
print(f"진료과 관련 컬럼들에 산부인과 또는 정신건강의학과가 없는가? → {no_obgyn_or_psych}")


In [ ]:
# Dx_KCD 확인

import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Dx-KCD/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "**/*.csv"), recursive=True)

# ① 전체 고유 환자 ID 저장
unique_pt_ids = set()

# ② 진료과 관련 키워드 존재 여부
no_obgyn_or_psych = True  # True면 없음
target_cols = ['depart', 'ad_depart_nm', 'dc_depart_nm']
keywords = "산부인과|정신건강의학과"

# ③ 상병명에서 'Deidentified' 값 개수
deidentified_count = 0

# 파일 탐색
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype=str)

        # pt_id 수집
        if 'pt_id' in table_df.columns:
            unique_pt_ids.update(table_df['pt_id'].dropna().unique())

        # 진료과 관련 키워드 포함 여부
        if no_obgyn_or_psych:
            for col in target_cols:
                if col in table_df.columns:
                    if table_df[col].fillna('').str.contains(keywords, regex=True).any():
                        no_obgyn_or_psych = False
                        break

        # 상병명에서 'Deidentified' 개수
        if 'dig_nm' in table_df.columns:
            deidentified_count += (table_df['dig_nm'].fillna('') == 'Deidentified').sum()

    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")

# ✅ 최종 결과 출력
print("\n✅ [Dx_KCD 종합 결과]")
print(f"전체 고유 환자 수: {len(unique_pt_ids)}")
print(f"진료과 컬럼들에 산부인과 또는 정신건강의학과가 없는가? → {no_obgyn_or_psych}")
print(f"'Deidentified'로 표시된 상병명 개수: {deidentified_count}")


In [ ]:
# Flowsheet 확인
import os
import glob
import pandas as pd

# Flowsheet 폴더 경로
flowsheet_path = str(PROCESSED_DIR / 'AEGIS-Flowsheet-Value/de')
csv_files = glob.glob(os.path.join(flowsheet_path, '**/*.csv'), recursive=True)

all_patient_ids = set()
non_numeric_values_summary = []

for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype={'pt_id': str}, sep=',', quotechar='"', lineterminator='\n')

        # 환자 ID 수집
        if 'pt_id' in table_df.columns:
            all_patient_ids.update(table_df['pt_id'].dropna().unique())

        # cat_value 검사
        if 'cat_value' in table_df.columns:
            cat_values = table_df['cat_value'].dropna().astype(str)

            # 숫자 or Y/N 필터링
            valid_mask = (
                cat_values.str.upper().isin(['Y', 'N']) |
                cat_values.str.match(r'^-?\d+(\.\d+)?$', na=False)
            )

            invalid_values = cat_values[~valid_mask].unique().tolist()

            if invalid_values:
                non_numeric_values_summary.append({
                    'file': os.path.basename(file),
                    'invalid_values': invalid_values
                })

    except Exception as e:
        print(f"❌ 오류 발생: {file} -> {e}")

# 결과 요약
print(f"✅ Flowsheet 전체 환자 수: {len(all_patient_ids)}명")

if non_numeric_values_summary:
    print("\n⚠️ 수치형/Y/N 이외의 값이 존재하는 파일:")
    for entry in non_numeric_values_summary:
        print(f"📂 {entry['file']} → {entry['invalid_values']}")
else:
    print("🎉 모든 파일에서 cat_value는 수치형, Y, N 값만 존재합니다.")


In [ ]:
# Administer 확인

import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Administer-Med/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "**/*.csv"), recursive=True)

# ① 전체 고유 환자 ID 저장
unique_pt_ids = set()

# ② 진료과 관련 키워드 존재 여부
no_obgyn_or_psych = True  # True면 없음
target_cols = ['depart_nm']
keywords = "산부인과|정신건강의학과"

# 파일 탐색
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype=str)

        # pt_id 수집
        if 'pt_id' in table_df.columns:
            unique_pt_ids.update(table_df['pt_id'].dropna().unique())

        # 진료과 관련 키워드 포함 여부
        if no_obgyn_or_psych:
            for col in target_cols:
                if col in table_df.columns:
                    if table_df[col].fillna('').str.contains(keywords, regex=True).any():
                        no_obgyn_or_psych = False
                        break

    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")

# ✅ 최종 결과 출력
print("\n✅ [Administer 종합 결과]")
print(f"전체 고유 환자 수: {len(unique_pt_ids)}")
print(f"진료과 컬럼들에 산부인과 또는 정신건강의학과가 없는가? → {no_obgyn_or_psych}")


In [ ]:
# Basic_order 확인

import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Basic-Order/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "**/*.csv"), recursive=True)

# ① 전체 고유 환자 ID 저장
unique_pt_ids = set()

# ② 진료과 관련 키워드 존재 여부
no_obgyn_or_psych = True  # True면 없음
target_cols = ['depart_nm']
keywords = "산부인과|정신건강의학과"

# 파일 탐색
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        # pt_id 수집
        if 'pt_id' in table_df.columns:
            unique_pt_ids.update(table_df['pt_id'].dropna().unique())

        # 진료과 관련 키워드 포함 여부
        if no_obgyn_or_psych:
            for col in target_cols:
                if col in table_df.columns:
                    if table_df[col].fillna('').str.contains(keywords, regex=True).any():
                        no_obgyn_or_psych = False
                        break

    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")

# ✅ 최종 결과 출력
print("\n✅ [Basic_order 종합 결과]")
print(f"전체 고유 환자 수: {len(unique_pt_ids)}")
print(f"진료과 컬럼들에 산부인과 또는 정신건강의학과가 없는가? → {no_obgyn_or_psych}")


In [ ]:
# Order_med 확인

import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Order-Med1/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "**/*.csv"), recursive=True)

# ① 전체 고유 환자 ID 저장
unique_pt_ids = set()


# 파일 탐색
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        # pt_id 수집
        if 'pt_id' in table_df.columns:
            unique_pt_ids.update(table_df['pt_id'].dropna().unique())


    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")

# ✅ 최종 결과 출력
print("\n✅ [Order_med 종합 결과]")
print(f"전체 고유 환자 수: {len(unique_pt_ids)}")


In [ ]:
# Order_test 확인

import os
import glob
import pandas as pd

# 경로 설정
processed_table_dir = str(PROCESSED_DIR / 'AEGIS-Order-Test/de')
csv_files = glob.glob(os.path.join(processed_table_dir, "**/*.csv"), recursive=True)

# ① 전체 고유 환자 ID 저장
unique_pt_ids = set()


# 파일 탐색
for file in csv_files:
    try:
        table_df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')

        # pt_id 수집
        if 'pt_id' in table_df.columns:
            unique_pt_ids.update(table_df['pt_id'].dropna().unique())


    except Exception as e:
        print(f"❌ 오류 발생 - {file}: {e}")

# ✅ 최종 결과 출력
print("\n✅ [AEGIS-Order-Test 종합 결과]")
print(f"전체 고유 환자 수: {len(unique_pt_ids)}")
